# Chapter 1 — Python OOP Fundamentals

This notebook starts from the absolute basics of Python OOP, but every concept will be tied back to a bigger question:

> How does this help you build a reusable, platform-independent agent?

In this chapter, we focus on:

- Classes
- Objects
- Instances
- Attributes
- Methods
- Constructors
- `self`

We will first use a tiny non-AI example, then translate the same ideas into an agentic example.

## 1. Concept

### What is a class?
A **class** is a blueprint. It describes what data an object should hold and what behavior it should have.

### What is an object?
An **object** is a real thing created from a class.

### What is an instance?
In Python, an object created from a class is also called an **instance** of that class.

### What are attributes?
Attributes are the data stored on an object.

### What are methods?
Methods are functions defined inside a class. They describe what the object can do.

### What is a constructor?
The constructor in Python is usually the `__init__` method. It runs when you create an object and helps initialize its data.

### What is `self`?
`self` refers to the current object instance. It lets one object access its own attributes and methods.

At this stage, you can think of a class as answering two questions:

1. What data should this thing remember?
2. What actions should this thing perform?

In [1]:
# A tiny warm-up to keep the notebook executable
print('Chapter 1 loaded: Python OOP Fundamentals')

Chapter 1 loaded: Python OOP Fundamentals


## 2. Simple Example

Let us start with a very small class that has nothing to do with AI.

In [ ]:
class Calculator:
# here __init__ is a constructor method that initializes the brand attribute of the Calculator class
    def __init__(self, brand: str) -> None:
        self.brand = brand

    def add(self, a: float, b: float) -> float:
        return a + b

    def multiply(self, a: float, b: float) -> float:
        return a * b

In [ ]:
calculator = Calculator(brand='Casio')

print('Type of object:', type(calculator).__name__)
print('Calculator brand:', calculator.brand)
print('2 + 3 =', calculator.add(2, 3))
print('4 * 5 =', calculator.multiply(4, 5))

### Breaking this down

- `Calculator` is the **class**.
- `calculator = Calculator(...)` creates an **object**.
- That object is an **instance** of `Calculator`.
- `brand` is an **attribute**.
- `add` and `multiply` are **methods**.
- `__init__` is the **constructor**.
- `self.brand = brand` stores data on the current object.

## 3. Why `self` Matters

A beginner confusion is: why do we need `self` at all?

Because each object should hold its own state. `self` is how Python lets one object refer to its own data.

In [ ]:
calculator_a = Calculator('Casio')
calculator_b = Calculator('Texas Instruments')

print(calculator_a.brand)
print(calculator_b.brand)

Even though both objects come from the same class, they can store different values.

That is one of the most important ideas in OOP: many objects can be created from one blueprint, but each object has its own state.

## 4. Agentic Example

Now let us move from `Calculator` to a very small `Agent`.

This is still intentionally simple. We are not trying to build a full architecture yet. We are only using the agent example to understand classes and objects.

In [ ]:
class Agent:
    def __init__(self, name: str, role: str) -> None:
        self.name = name
        self.role = role

    def describe(self) -> str:
        return f'{self.name} is an agent responsible for {self.role}.'

    def respond(self, question: str) -> str:
        return f'{self.name} received: {question}'

In [ ]:
agent = Agent(name='InsightAgent', role='answering business questions')

print(agent.describe())
print(agent.respond('What was Q3 revenue?'))

### What is happening here?

- `Agent` is a class.
- `agent` is an object / instance.
- `name` and `role` are attributes.
- `describe` and `respond` are methods.
- `__init__` constructs the object.
- `self.name` and `self.role` store instance-specific state.

## 5. Why This Matters for Agentic Systems

At first glance, this may feel basic, but this is where architecture begins.

If you cannot clearly say:

- what an object is responsible for,
- what data it owns, and
- what behavior it provides,

then larger architecture becomes messy very quickly.

When you build an agentic system, you will eventually need objects such as:

- `Agent`
- `Tool`
- `ToolRegistry`
- `LLMClient`
- `DataProvider`
- `QueryService`
- `QueryRequest`
- `QueryResult`

OOP gives you a way to represent these responsibilities clearly instead of putting everything into one huge script.

## 6. Important Architectural Insight

An agent should usually **not** know how the underlying data platform works.

For example, the agent may need the capability: "answer a revenue question".

But that does **not** mean the agent should know:

- how to connect to Snowflake,
- how Databricks exposes a metrics API,
- how AWS Athena expects query execution,
- or what raw SDK response objects look like.

In good architecture, the agent object should represent **agent behavior**, not **platform plumbing**.

We are not solving that fully yet, but this chapter gives you the vocabulary to even discuss that design properly.

## 7. Bad Implementation

Here is a deliberately bad example. It still uses a class, but it mixes too many responsibilities.

In [ ]:
class BadAgent:
    def __init__(self, name: str) -> None:
        self.name = name

    def answer_revenue_question(self) -> str:
        # Pretend these are platform-specific details
        host = 'snowflake.mycompany.com'
        warehouse = 'FINANCE_WH'
        sql = "SELECT revenue FROM quarterly_metrics WHERE quarter = 'Q3'"

        return (
            f'{self.name} connected to {host}, used warehouse {warehouse}, '
            f'ran SQL: {sql}'
        )

In [ ]:
bad_agent = BadAgent('InsightAgent')
print(bad_agent.answer_revenue_question())

### Why is this bad?

Because the class is supposed to represent an agent, but it also contains data-platform details.

That creates problems:

- Harder to switch platforms
- Harder to test
- Harder to reuse
- Harder to understand
- Agent logic and infrastructure logic become tangled

## 8. Refactor into a Better Early Design

We are still early, so we will not introduce interfaces or dependency injection yet. But we can already separate responsibilities a little.

In [ ]:
class RevenueService:
    def get_q3_revenue(self) -> str:
        # Still fake, but now platform work is no longer inside Agent
        sql = "SELECT revenue FROM quarterly_metrics WHERE quarter = 'Q3'"
        return f'Executed query: {sql}'


class BetterAgent:
    def __init__(self, name: str, revenue_service: RevenueService) -> None:
        self.name = name
        self.revenue_service = revenue_service

    def answer_revenue_question(self) -> str:
        result = self.revenue_service.get_q3_revenue()
        return f'{self.name} says: {result}'

In [ ]:
revenue_service = RevenueService()
better_agent = BetterAgent('InsightAgent', revenue_service)
print(better_agent.answer_revenue_question())

### Why is this better?

The improvement is small but meaningful:

- `BetterAgent` focuses more on agent behavior.
- `RevenueService` handles revenue retrieval logic.
- Responsibilities are starting to separate.

This is not yet the final architecture, but it points in the right direction.

## 9. Trade-offs

Even this early refactor has trade-offs.

### Benefits
- Cleaner responsibilities
- Easier to reason about
- Easier to expand later

### Costs
- More classes
- Slightly more indirection
- Can feel unnecessary for tiny scripts

This is an important lesson for the whole journey: do not add abstraction just because it sounds sophisticated. Add it when it makes responsibilities clearer or future changes safer.

## 10. When NOT to Overdo OOP

You should not create a class for every tiny thing.

For example, if you only need a one-line helper function used once, a class may be unnecessary.

Use classes when:

- you need to group related data and behavior,
- you want reusable objects,
- the concept has identity or state,
- or you are modeling important responsibilities.

Do not use classes just to make code look ‘enterprise’.

## 11. Exercise

Try this yourself before reading the solution.

### Exercise 1
Create a class called `ReportFormatter` with:

- an attribute `format_name`
- a constructor to set it
- a method `format_title(title: str) -> str`

Then create an object and call the method.

### Exercise 2
Create a class called `SimpleAgent` with:

- `name`
- `domain`
- a method `introduce()`

Then instantiate two different agents and print their introductions.

## 12. Solution


In [ ]:
class ReportFormatter:
    def __init__(self, format_name: str) -> None:
        self.format_name = format_name

    def format_title(self, title: str) -> str:
        return f'[{self.format_name}] {title}'


formatter = ReportFormatter('UPPERCASE_STYLE')
print(formatter.format_title('Quarterly Revenue Report'))


class SimpleAgent:
    def __init__(self, name: str, domain: str) -> None:
        self.name = name
        self.domain = domain

    def introduce(self) -> str:
        return f'I am {self.name}, and I work on {self.domain}.'


agent_a = SimpleAgent('FinanceAgent', 'finance analytics')
agent_b = SimpleAgent('SupportAgent', 'customer support')

print(agent_a.introduce())
print(agent_b.introduce())

## 13. Takeaway

The key lesson from this chapter is simple:

- A **class** is a blueprint.
- An **object** is an instance of that blueprint.
- **Attributes** store state.
- **Methods** define behavior.
- `__init__` initializes an object.
- `self` lets an object refer to its own state and behavior.

And most importantly for your larger goal:

Good architecture starts by assigning clear responsibilities to objects.

If the `Agent` object tries to do everything itself, the design becomes hard to test, hard to switch across platforms, and hard to maintain.

In the next chapter, we can build on this by learning **encapsulation** — how objects hide internal details and expose only what other parts of the system should use.